# Tugas Kelompok 2: Data Collection Pipeline

## Kelompok sowang

1. Fransisca Ellya Bunaren (2306152286)
2. Micheline Wijaya Limbergh (2306207013)
3. Nisrina Annaisha Sarnadi (2306275960)
4. Rizqya Az Zahra Putri (2306244936)
5. Muhammad Satrio Haryo Kusumo (2006597771)

# Preprocessing untuk Topic Modeling (RQ1) dan Temporal Analysis (RQ3)

Tahap ini bertujuan menghasilkan dataset postingan yang relevan dengan pembahasan *parenting* dari data hasil cleaning. Data yang dikumpulkan melalui pencarian berbasis kata kunci masih memuat postingan yang tidak membahas pengasuhan anak secara substantif, seperti promosi produk atau kelas, spam, konten dewasa, dan pembahasan acara atau artis yang kebetulan memuat kata kunci yang sama. Postingan semacam ini perlu disaring agar tidak mengganggu analisis topik, sentimen, dan tren pada tahap berikutnya.


## 0. Konfigurasi, kamus slang, dan stopword

In [1]:
from pathlib import Path
import re, json, html, unicodedata, urllib.request
from collections import Counter
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display

CURRENT = Path.cwd().resolve()
ROOT = CURRENT if (CURRENT / "data" / "labeling").exists() else CURRENT.parent
LABEL_DIR = ROOT / "data" / "labeling"
_cands = [LABEL_DIR / n for n in ("relevant_final.csv", "relevant.csv")]
INPUT = next((p for p in _cands if p.exists()), _cands[-1])
OUT_DIR = ROOT / "data" / "processed" / "rq1_3" 
OUT_DIR.mkdir(parents=True, exist_ok=True)
EXT_DIR = ROOT / "data" / "external"
LEXICON_CSV = EXT_DIR / "colloquial-indonesian-lexicon.csv"     
LEXICON_URL = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
SLANG_EXTRA_CSV = OUT_DIR / "slang_extra.csv"                   

# Pengaturan
SEED = 2026
MIN_TOKENS = 5            
MIN_CONTENT = 3           
MIN_LDA_TOKENS = 3        
USE_STEMMING = True
USE_BIGRAMS = True
BIGRAM_MIN_COUNT = 20
BIGRAM_THRESHOLD = 0.5
USE_LEXICON = True        
USE_AFFIX_RULES = True    
pd.set_option("display.max_colwidth", 140)

def read_csv_str(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")

# kata yang dianggap "dikenal" saat mengukur cakupan
KNOWN_EXTRA = {"dulu", "ponsel", "tiktok", "instagram", "facebook", "youtube", "whatsapp", "gadget", "online", "video",
               "internet", "update", "medsos", "trauma", "mental", "emosi", "disiplin", "stres", "konflik"}

# kamus slang
SLANG = {
    # negasi & penghubung
    "gak": "tidak", "ga": "tidak", "nggak": "tidak", "ngga": "tidak", "enggak": "tidak", "engga": "tidak",
    "gk": "tidak", "tdk": "tidak", "tak": "tidak", "nggk": "tidak", "kagak": "tidak", "ndak": "tidak", "nda": "tidak",
    "gaada": "tidak ada", "gada": "tidak ada",
    "yg": "yang", "yng": "yang", "dgn": "dengan", "dg": "dengan", "dng": "dengan",
    "krn": "karena", "karna": "karena", "krna": "karena", "tp": "tapi", "tpi": "tapi", "jd": "jadi", "jdi": "jadi",
    "utk": "untuk", "tuk": "untuk", "bwt": "buat", "dr": "dari", "drpd": "daripada", "dripada": "daripada",
    "sm": "sama", "ama": "sama", "sbg": "sebagai", "dlm": "dalam", "kalo": "kalau", "klo": "kalau", "kl": "kalau",
    "klau": "kalau", "kalok": "kalau", "sampe": "sampai", "smpe": "sampai", "pdhl": "padahal", "jgn": "jangan",
    "blm": "belum", "belom": "belum", "blum": "belum", "spt": "seperti", "sprti": "seperti", "kek": "seperti",
    "sblm": "sebelum", "stlh": "setelah",
    # waktu & penekanan
    "udah": "sudah", "udh": "sudah", "dah": "sudah", "sdh": "sudah", "uda": "sudah", "wes": "sudah",
    "lg": "lagi", "lgi": "lagi", "trs": "terus", "trus": "terus", "mulu": "terus", "bgt": "sangat", "bngt": "sangat",
    "banget": "sangat", "bgd": "sangat", "aja": "saja", "aj": "saja", "sj": "saja", "doang": "saja", "doank": "saja",
    "cm": "hanya", "cuma": "hanya", "cuman": "hanya", "cmn": "hanya", "hny": "hanya",
    "emg": "memang", "emang": "memang", "mmg": "memang", "gmn": "bagaimana", "gimana": "bagaimana",
    "gmna": "bagaimana", "gmana": "bagaimana", "knp": "kenapa", "bs": "bisa", "bsa": "bisa", "bkn": "bukan",
    "bukn": "bukan", "hrs": "harus", "mesti": "harus", "musti": "harus", "kudu": "harus",
    "gitu": "begitu", "gini": "begini", "gt": "begitu", "lbh": "lebih", "kyk": "seperti", "kyknya": "sepertinya",
    "kayaknya": "sepertinya", "mnrt": "menurut", "tau": "tahu", "pake": "pakai", "dipake": "dipakai",
    "thn": "tahun", "bln": "bulan", "hr": "hari", "skrg": "sekarang", "skrng": "sekarang", "sekrg": "sekarang",
    "kmrn": "kemarin", "bsk": "besok", "jaman": "zaman", "abis": "habis", "msh": "masih", "masi": "masih",
    "smua": "semua", "byk": "banyak", "bnyk": "banyak", "dikit": "sedikit", "sdkt": "sedikit", "krg": "kurang",
    "kurg": "kurang", "bnr": "benar", "bener": "benar", "dpt": "dapat", "blh": "boleh", "bole": "boleh",
    "mgkn": "mungkin", "mngkn": "mungkin", "sbnrnya": "sebenarnya", "sebenernya": "sebenarnya",
    "mw": "mau", "pengen": "ingin", "pingin": "ingin", "pengin": "ingin", "pgn": "ingin", "pngn": "ingin",
    # kata ganti & orang
    "sy": "saya", "sya": "saya", "saia": "saya", "aq": "saya", "gue": "saya", "gua": "saya", "gw": "saya",
    "aku": "saya", "km": "kamu", "kmu": "kamu", "lu": "kamu", "lo": "kamu", "dy": "dia", "doi": "dia",
    "mrk": "mereka", "tmn": "teman", "temen": "teman",
    "ortu": "orang tua", "orgtua": "orang tua", "orangtua": "orang tua", "org": "orang", "ank": "anak",
    "bocil": "anak kecil", "nak": "anak", "dedek": "adik", "dede": "adik", "adek": "adik", "kk": "kakak",
    "nyokap": "ibu", "bokap": "ayah", "emak": "ibu", "mak": "ibu", "mamah": "mama", "papah": "papa",
    "bund": "bunda", "bunds": "bunda", "cowok": "laki-laki", "cewek": "perempuan",
    "klrga": "keluarga", "klg": "keluarga", "kluarga": "keluarga", "sklh": "sekolah", "skolah": "sekolah",
    "bumil": "ibu hamil", "busui": "ibu menyusui", "abg": "remaja",
    "dll": "dan lain lain", "dsb": "dan sebagainya", "bgs": "bagus", "kdrt": "kekerasan dalam rumah tangga",
    # perasaan & kata kerja kolokial yang sering muncul
    "baper": "bawa perasaan", "mager": "malas", "gabut": "menganggur", "gaje": "tidak jelas", "santuy": "santai",
    "kepo": "penasaran", "lebay": "berlebihan", "bete": "kesal", "kesel": "kesal", "sebel": "sebal",
    "ngeselin": "menyebalkan", "nyebelin": "menyebalkan", "cape": "lelah", "capek": "lelah",
    "nyesel": "menyesal", "kasian": "kasihan", "maap": "maaf", "ngambek": "merajuk", "ngeyel": "membantah",
    "ngelawan": "melawan", "nurut": "menurut", "nurutin": "menuruti", "ngerti": "mengerti", "ngerasa": "merasa",
    "ngomong": "berbicara", "ngobrol": "mengobrol", "nanya": "bertanya", "nanyain": "menanyakan",
    "liat": "lihat", "ngeliat": "melihat", "nyuruh": "menyuruh", "nyari": "mencari", "ngatur": "mengatur",
    "ngurus": "mengurus", "ngurusin": "mengurus", "ngajak": "mengajak", "ngasih": "memberi", "ngebentak": "membentak",
    "ngomel": "mengomel", "mikir": "berpikir", "mikirin": "memikirkan", "pikirin": "memikirkan",
    "ngedidik": "mendidik", "ngajarin": "mengajarkan", "ngapain": "mengapa", "bikin": "buat", "bikinin": "membuatkan",
    "banding2in": "membandingkan", "bandingin": "membandingkan", "dibandingin": "dibandingkan",
    "dimarahin": "dimarahi", "dikasih": "diberi", "kasih": "kasih", "hp": "ponsel", "hape": "ponsel",
    "medsos": "media sosial", "ig": "instagram", "fb": "facebook", "yt": "youtube", "wa": "whatsapp",
    "yaudah": "ya sudah", "yauda": "ya sudah", "yowes": "ya sudah",
}

# stopword
try:
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
except ImportError as e:
    raise ImportError("Paket belum terpasang. Jalankan di terminal: pip install Sastrawi") from e
STOP_ID = set(StopWordRemoverFactory().get_stop_words())
STOP_ID_EXTRA = {"sangat", "saya", "kamu", "nih", "tuh", "deh", "dong", "kok", "sih", "lho", "loh", "kan",
                 "yah", "ya", "yaa", "nya", "kayak", "kali", "mau", "tapi", "hanya", "saja", "memang", "begitu",
                 "begini", "bagaimana", "kenapa", "terus", "banyak", "sama", "lebih", "seperti", "pakai",
                 "tahu", "jadi", "lagi", "sudah", "tidak", "bukan", "jangan", "harus", "bisa", "karena",
                 "kalau", "buat", "apa", "punya", "biar", "pas", "si", "malah", "padahal", "pernah", "soal", "kaya",
                 "ingin", "sepertinya", "enggak", "kalo", "noh", "nah", "lah", "kah", "pun",
                 "anjir", "anjay", "njir", "bjir", "buset", "astaga", "duh", "aduh", "waduh", "hadeh", "halah",
                 "huhu", "hiks", "hmm", "yaelah", "gaes", "guys", "gais", "kak", "mba", "mbak", "mas", "bang",
                 "bund", "bunda", "bu", "pak"}
try:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    STOP_EN = set(ENGLISH_STOP_WORDS)
except ImportError:
    STOP_EN = {"the", "a", "an", "and", "or", "but", "is", "are", "was", "were", "be", "to", "of", "in", "on",
               "for", "with", "it", "this", "that", "i", "you", "he", "she", "we", "they", "my", "your", "not", "so"}
DOMAIN_WORDS = {"parenting"}
DOMAIN_PHRASE_RE = re.compile(r"\bpola asuh\b")
STOP = STOP_ID | STOP_ID_EXTRA | STOP_EN | DOMAIN_WORDS

# query
QUERY_TERMS = {"gentle", "positive", "respectful", "strict", "authoritarian", "authoritative", "permissive",
               "helicopter", "attachment", "intensive", "voc", "otoriter", "permisif", "demokratis"}

_factory = StemmerFactory()
LEMMAS = set(_factory.get_words())
stemmer = _factory.create_stemmer()
STEM_FIX = {"pengalaman": "pengalaman", "setuju": "setuju", "perasaan": "rasa", "bentakan": "bentak",
            "perhatian": "perhatian", "lingkungan": "lingkungan", "mengerti": "mengerti", "pendapat": "pendapat"}
_stem_cache = {}
def stem_word(w):
    if w in STEM_FIX:
        return STEM_FIX[w]
    if w not in _stem_cache:
        _stem_cache[w] = stemmer.stem(w)
    return _stem_cache[w]

n_curated = len(SLANG)
n_extra = 0
if SLANG_EXTRA_CSV.exists():
    ex = read_csv_str(SLANG_EXTRA_CSV)
    ex = ex[(ex["slang"].str.strip() != "") & (ex["baku"].str.strip() != "")]
    EXTRA = {s.strip().lower(): b.strip().lower() for s, b in zip(ex["slang"], ex["baku"])}
    SLANG.update(EXTRA)                      
    n_extra = len(EXTRA)

LEX = {}
if USE_LEXICON:
    if not LEXICON_CSV.exists():
        try:
            EXT_DIR.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(LEXICON_URL, LEXICON_CSV)
            print("Leksikon slang diunduh ke", LEXICON_CSV)
        except Exception as e:
            print(f"Leksikon tidak bisa diunduh ({type(e).__name__}). Lanjut hanya dengan kamus sendiri. "
                  f"Unduh manual dari {LEXICON_URL} lalu simpan di {LEXICON_CSV}")
    if LEXICON_CSV.exists():
        lx = read_csv_str(LEXICON_CSV)[["slang", "formal"]]
        lx["slang"], lx["formal"] = lx["slang"].str.lower().str.strip(), lx["formal"].str.lower().str.strip()
        lx = lx[lx["slang"].str.fullmatch(r"[a-z]{3,}") & lx["formal"].ne("") & lx["slang"].ne(lx["formal"])]
        lx = lx[~lx["slang"].isin(LEMMAS) & ~lx["slang"].isin(STOP) & ~lx["slang"].isin(SLANG)]
        LEX = lx.groupby("slang")["formal"].agg(lambda s: s.value_counts().index[0]).to_dict()

def _canon(w):
    return SLANG.get(w, w)

def _lookup(w):
    if w in SLANG:
        return SLANG[w]
    if w in LEX:
        return _canon(LEX[w])
    return w

# buang awakan kolokial
def _prefix_base(c):
    if c in LEMMAS:
        return c
    if c.startswith("nge") and len(c) > 5 and c[3:] in LEMMAS:        
        return c[3:]
    if c.startswith("ng") and len(c) > 3 and c[2] in "aiueo":           
        for pre in ("k", ""):
            if pre + c[2:] in LEMMAS:
                return pre + c[2:]
    if c.startswith("ny") and len(c) > 3 and c[2] in "aiueo":           
        for pre in ("s", "c"):
            if pre + c[2:] in LEMMAS:
                return pre + c[2:]
    if c.startswith("n") and len(c) > 3 and c[1] in "aiueo" and "t" + c[1:] in LEMMAS:   
        return "t" + c[1:]
    if c.startswith("m") and len(c) > 3 and c[1] in "aiueo" and "p" + c[1:] in LEMMAS:   
        return "p" + c[1:]
    return None

# normalisasi kata
def normalize_word(w):
    r = _lookup(w)
    if r != w:
        return r
    if not USE_AFFIX_RULES or len(w) < 4 or w in LEMMAS or w in STOP:
        return w
    if w.endswith("nya") and len(w) > 5:
        b = _lookup(w[:-3])
        if b != w[:-3]:
            return b + "nya"
    passive = w.startswith("di") and w.endswith("in") and len(w) > 6    
    active = (not passive) and w.endswith("in") and len(w) > 5        
    core = w[2:-2] if passive else w[:-2] if active else w
    base = _prefix_base(core)
    if base is None or base in STOP or len(base) < 3:
        return w
    return ("di" + base + "i") if passive else (base + "kan") if active else base

print("Project root:", ROOT, "\nInput       :", INPUT, "\nOutput      :", OUT_DIR)
print(f"Slang: kamus awal {n_curated} | slang_extra.csv {n_extra} | leksikon publik {len(LEX):,} entri "
      f"| aturan imbuhan {'aktif' if USE_AFFIX_RULES else 'mati'}")
print(f"Stopword: {len(STOP)} kata | kata dasar baku: {len(LEMMAS):,}")

Project root: C:\Users\nisri\OneDrive\Documents\University\Anmedsos\sowang 
Input       : C:\Users\nisri\OneDrive\Documents\University\Anmedsos\sowang\data\labeling\relevant_final.csv 
Output      : C:\Users\nisri\OneDrive\Documents\University\Anmedsos\sowang\data\processed\rq1_3
Slang: kamus awal 253 | slang_extra.csv 0 | leksikon publik 3,552 entri | aturan imbuhan aktif
Stopword: 516 kata | kata dasar baku: 29,933


## 1. Input `relevant.csv`

In [2]:
df = read_csv_str(INPUT)
print("Membaca:", INPUT.name)
if INPUT.name == "relevant.csv":
    print("Memakai relevant.csv. Dataset akhir dari 02 adalah relevant_v2.csv.")

# filter kolom yang digunakan
need = ["id", "text", "created_at", "source_month", "group", "is_reply"]        
missing = [c for c in need if c not in df.columns]
assert not missing, f"Kolom wajib hilang di {INPUT.name}: {missing}"
assert df["id"].is_unique, f"ID tidak unik di {INPUT.name}"

# sinkronisasi dengan output labeling
rep_path = LABEL_DIR / "labeling_report.json"
rem_path = LABEL_DIR / "removed_relevance.csv"
if rep_path.exists():
    n_cand = json.load(open(rep_path, encoding="utf-8")).get("candidate_relevant")
    if INPUT.name == "relevant.csv":
        if n_cand is not None:
            assert len(df) == n_cand, f"relevant.csv ({len(df):,}) != labeling_report.json ({n_cand:,}). Jalankan ulang 02."
    else:
        ids_02 = set(read_csv_str(LABEL_DIR / "relevant.csv")["id"])
        if rem_path.exists():
            ids_02 |= set(read_csv_str(rem_path)["id"])
        assert set(df["id"]) <= ids_02, f"Ada ID di {INPUT.name} yang tidak berasal dari output 02. Jalankan ulang 02."
        print(f"Dataset akhir: {len(df):,} baris (kandidat otomatis di laporan 02: {n_cand if n_cand is not None else '-'})")
else:
    print("labeling_report.json tidak ada: sinkronisasi dengan notebook 02 tidak bisa diperiksa.")

# validasi timestamp
ts = pd.to_datetime(df["created_at"], utc=True, errors="coerce")
assert ts.notna().all(), f"{int(ts.isna().sum())} baris created_at tidak bisa dibaca sebagai tanggal"
month_mismatch = ts.dt.strftime("%Y-%m") != df["source_month"]
if month_mismatch.any():
    print(f"{int(month_mismatch.sum())} baris: bulan created_at (UTC) berbeda dari source_month. Periksa sebelum analisis temporal.")
    display(df.loc[month_mismatch, ["id", "created_at", "source_month"]].head(5))
else:
    print("Timestamp valid")
print(f"Rentang waktu: {ts.min():%Y-%m-%d} - {ts.max():%Y-%m-%d}")

print(f"Baris dimuat: {len(df):,} | bulan: {df['source_month'].nunique()} | kolom: {df.shape[1]}")
display(df[["id", "source_month", "group", "text"]].head(5))

Membaca: relevant_final.csv
Dataset akhir: 8,072 baris (kandidat otomatis di laporan 02: 8078)
Timestamp valid
Rentang waktu: 2025-10-01 - 2026-09-30
Baris dimuat: 8,072 | bulan: 12 | kolom: 27


,id,source_month,group,text
0,1973219344600760446,2025-10,umum,"Lebih butuh konseling pra- nikah yg mendalam, bkn sekedar yel”\nIdealnya ada ahli konseling keluarga, psikolog atau perencana keuangan y..."
1,1973223454599262597,2025-10,umum,@toe_giman Miris... Akibat dr pny ortu tp ga bs bikin teladan yg baik dn bnr bwt anak2ny. Yg blm nikah yuk prsiapkn dr klian utk pnya il...
2,1973230829380968860,2025-10,otoriter,@sopir_idiot Kalau kayak gini. Ngasitaunya harus pakai gentle parenting atau VOC parenting?\nSerius nanya dengan nada selembut ulat sutra
3,1973234406467969447,2025-10,umum,@kitoorah @tanyakanrl mau pemerintah bikin aturan gmnpun kalo emng parentingny ga bagus sama aja. urusan kayak gini bukan karna pemrnth...
4,1973234962532045079,2025-10,umum,"Nanti deh mau ngumpulin video2 xinlong yg kusuka, video-video tentang bagusnya attitude dia\n\nBtw mama xinlong tidak mau buka kelas par..."


## 2. Preprocess untuk kedua rencana model

| # | Langkah | Contoh |
|---|---|---|
| 1 | Unicode, `&amp;`, spasi | `anak &amp; ibu` -> `anak & ibu` |
| 2 | Buang URL dan mention | `@budi lihat https://t.co/x` -> `lihat` |
| 3 | Pisahkan tagar CamelCase, lalu huruf kecil | `#PolaAsuhOtoriter` -> `pola asuh otoriter` |
| 4 | Tagar: buang `#`, kata dipertahankan | `#gentleparenting` -> `gentle parenting` |
| 5 | Huruf berulang (≥3) jadi satu | `bangeeet` -> `banget` |
| 6 | Buang emoji, angka, tanda baca (tanda hubung dalam kata dipertahankan) | `anak-anak 😭!!` -> `anak-anak` |
| 7 | Slang → baku (4 lapis, lihat Bagian 0), tawa (`wkwk`, `haha`) dibuang | `yg gak bgt` -> `yang tidak sangat` |

In [3]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
CAMEL_RE = re.compile(r"(?<=[a-z])(?=[A-Z])")
GLUE_RE = re.compile(r"\b(gentle|positive|respectful|strict|authoritarian|authoritative|permissive|helicopter|attachment|intensive)parenting\b")
EMOJI_RE = re.compile("[\U0001F000-\U0001FAFF\u2190-\u21FF\u2300-\u23FF\u2460-\u27BF\u2900-\u297F\u2B00-\u2BFF"
                      "\uFE00-\uFE0F\u200D\u20E3\u00A9\u00AE\u203C\u2049\u2122\u2139]+")
LAUGH_RE = re.compile(r"^(?:wk)+\w*$|^(?:ha|he|hi)+h?$|^(?:kw)+k?$|^(?:xi)+x?$")

def clean_base(t):
    t = unicodedata.normalize("NFKC", str(t))                 
    t = html.unescape(t)                                      
    t = URL_RE.sub(" ", t)                                    
    t = MENTION_RE.sub(" ", t)                                
    t = HASHTAG_RE.sub(lambda m: " " + CAMEL_RE.sub(" ", m.group(1)) + " ", t)   
    t = t.lower()                                             
    t = GLUE_RE.sub(r"\1 parenting", t).replace("polaasuh", "pola asuh")
    t = re.sub(r"(.)\1{2,}", r"\1", t)                        
    t = EMOJI_RE.sub(" ", t)                                  
    t = re.sub(r"\d+", " ", t)                                
    t = re.sub(r"[^\w\s-]|_", " ", t)                         
    t = re.sub(r"(?<!\w)-+|-+(?!\w)", " ", t)                 
    t = re.sub(r"\b(?:nge|ng)-(?=\w)", "", t)                 
    words = []
    for w in t.split():                                       
        if LAUGH_RE.match(w):
            continue
        words.append(normalize_word(w))
    return " ".join(" ".join(words).split())

def content_tokens(t):
    t = DOMAIN_PHRASE_RE.sub(" ", t)
    return [w for w in t.split() if len(w) > 1 and w not in STOP and not w.startswith("parenting")]

df["text_embed"] = df["text"].map(clean_base)
tok_all = df["text_embed"].str.split()
df["n_tokens"] = tok_all.str.len()
df["n_content"] = df["text_embed"].map(lambda t: len(content_tokens(t)))

# data topic modeling
too_short = df["n_tokens"] < MIN_TOKENS
few_content = ~too_short & (df["n_content"] < MIN_CONTENT)
order = df.sort_values(["created_at", "id"], kind="stable").index
cand = order[(~too_short & ~few_content).loc[order].values]
dup_clean = pd.Series(False, index=df.index)
dup_clean.loc[cand] = df.loc[cand, "text_embed"].duplicated(keep="first").values

first_id = df.loc[cand].groupby("text_embed")["id"].first()
df["dup_of"] = np.where(dup_clean, df["text_embed"].map(first_id).fillna(""), "")

df["exclude_reason"] = np.select([too_short, few_content, dup_clean],
                                 ["too_short", "few_content", "dup_after_clean"], default="")
df["in_topic_corpus"] = df["exclude_reason"].eq("")

print(f"Masuk topic modeling : {df['in_topic_corpus'].sum():,} dari {len(df):,} ({df['in_topic_corpus'].mean():.1%})")
display(df.loc[~df["in_topic_corpus"], "exclude_reason"].value_counts().rename_axis("alasan").reset_index(name="jumlah"))

def retention(key, order=None):
    t = pd.DataFrame({"semua": df.groupby(key).size(), "ikut": df[df["in_topic_corpus"]].groupby(key).size()}).fillna(0).astype(int)
    t["ikut_%"] = (100 * t["ikut"] / t["semua"]).round(1)
    return t.reindex(order) if order else t
print("Per bulan:"); display(retention("source_month"))
print("Per kelompok:"); display(retention("group"))
if "is_reply" in df.columns:
    print("Reply vs bukan:"); display(retention("is_reply"))
spread = retention("source_month")["ikut_%"]
if spread.max() - spread.min() > 10:
    print(f"Persentase yang ikut berselisih {spread.max() - spread.min():.1f} poin antarbulan")

print("\nContoh post yang TIDAK ikut data:")
for reason in ["too_short", "few_content", "dup_after_clean"]:
    ex = df[df["exclude_reason"] == reason]
    if len(ex):
        print(f"\n[{reason}]")
        display(ex.sample(min(6, len(ex)), random_state=SEED)[["text", "text_embed"]])

Masuk topic modeling : 7,360 dari 8,072 (91.2%)


,alasan,jumlah
0,too_short,414
1,few_content,297
2,dup_after_clean,1


Per bulan:


,semua,ikut,ikut_%
source_month,,,
2025-10,678,616,90.9
2025-11,652,589,90.3
2025-12,660,583,88.3
2026-01,681,613,90.0
2026-02,684,624,91.2
2026-03,691,632,91.5
2026-04,689,643,93.3
2026-05,652,587,90.0
2026-06,655,605,92.4


Per kelompok:


...

,text,text_embed
6344,@EGA_82 VOC parenting yang bener,voc parenting yang benar
5275,Parenting yang aku suka😌 https://t.co/57QNgRtebC,parenting yang saya suka
31,@catlossnz aku di gentle parenting……,saya di gentle parenting
6314,@Audreyfqlne parenting jaman skrg,parenting zaman sekarang
7656,@tercyidukk Semua tergantung pola asuh,semua tergantung pola asuh
3924,voc parenting ya kit 😭😭😭 https://t.co/m1n78Ak4ms,voc parenting ya kit



[few_content]


,text,text_embed
3636,"@oyip_ Asian parenting, apa apa harus perfect 100 https://t.co/GalmWeLOge",asian parenting apa apa harus perfect
657,@sasmayaku Seru banget sih parenting yang kek ginj,seru sangat sih parenting yang seperti ginj
5146,Gak bisa gentle parenting ini https://t.co/1UzV3A2d93,tidak bisa gentle parenting ini
3613,parenting nya jelek banget najis https://t.co/2lY51OAYlw,parenting nya jelek sangat najis
7834,@Bersajak_id Gue suka bgt parenting beliau,saya suka sangat parenting beliau
7803,wkwk nyesel dia ga pake parenting voc😭😭 https://t.co/WFCg827SjY,menyesal dia tidak pakai parenting voc



[dup_after_clean]


,text,text_embed
64,#TKTKN\n24/09/2025 📸 TK Tunas Krida Nusantara menyelenggarakan pertemuan dengan orang tua siswa TK A dan TK B untuk kegiatan parenting d...,tktkn tk tunas krida nusantara menyelenggarakan pertemuan dengan orang tua siswa tk a dan tk b untuk kegiatan parenting dengan tema kema...


## 3. Preprocess untuk model LDA
Langkah: **stopword -> stemming -> frasa**.
- Stopword (Indonesia + Inggris + domain) dan kata 1 huruf dibuang.
- Stemming Sastrawi dijalankan **sekali per kata unik**, lalu hasilnya dipetakan kembali. Kata yang dipotong terlalu jauh dikoreksi lewat `STEM_FIX` (mis. `pengalaman` bukan `alam`); periksa kata terpopuler di Bagian 5 dan tambahkan bila perlu.
- Frasa (mis. `orang_tua`) dibentuk otomatis dengan gensim dari pasangan kata yang sering muncul bersama.
- `in_lda_corpus`: post yang ikut data **dan** punya ≥ `MIN_LDA_TOKENS` token. LDA lemah pada dokumen sangat pendek, jadi lihat distribusinya di bawah dan sesuaikan ambangnya.
- `text_tokens_noquery`: token yang sama **tanpa istilah pencarian** (`gentle`, `voc`, `otoriter`, ...). Istilah itu sengaja ada di sampel (kuota per kelompok), jadi bisa mendominasi topik; sediakan dua versi untuk perbandingan di notebook model.
- Ambang kosakata (`min_df`, `max_df`) **tidak** ditentukan di sini, tetapi di notebook model topik.

In [4]:
from gensim.models.phrases import Phrases
raw_tokens = df["text_embed"].map(content_tokens)

if USE_STEMMING:
    vocab = sorted({w for ts in raw_tokens for w in ts})
    print(f"Stemming {len(vocab):,} kata unik ...")
    stem_map = {w: stem_word(w) for w in vocab}
    tokens = raw_tokens.map(lambda ts: [s for s in (stem_map[w] for w in ts) if len(s) > 1 and s not in STOP])
else:
    tokens = raw_tokens

bigrams_found = []
if USE_BIGRAMS:
    phr = Phrases(tokens[df["in_topic_corpus"]].tolist(), min_count=BIGRAM_MIN_COUNT, threshold=BIGRAM_THRESHOLD, scoring="npmi")
    tokens = tokens.map(lambda ts: phr[ts])
    bigrams_found = sorted({w for ts in tokens for w in ts if "_" in w})
    print(f"Frasa terbentuk: {len(bigrams_found)} → {bigrams_found[:20]}")

tokens_nq = tokens.map(lambda ts: [w for w in ts if w not in QUERY_TERMS])
df["text_tokens"] = tokens.map(" ".join)
df["text_tokens_noquery"] = tokens_nq.map(" ".join)
df["n_lda_tokens"] = tokens.map(len)
df["n_lda_tokens_noquery"] = tokens_nq.map(len)
df["in_lda_corpus"] = df["in_topic_corpus"] & (df["n_lda_tokens"] >= MIN_LDA_TOKENS)

inc_ = df[df["in_topic_corpus"]]
print("\nDistribusi jumlah token LDA per post (data):")
display(inc_["n_lda_tokens"].describe().round(1).to_frame().T)
print(f"Dengan MIN_LDA_TOKENS = {MIN_LDA_TOKENS}: {int(df['in_lda_corpus'].sum()):,} post ikut LDA "
      f"({int((inc_['n_lda_tokens'] < MIN_LDA_TOKENS).sum()):,} dikeluarkan). Naikkan ambang bila median token rendah.")
print("Ambang lain (post data yang tersisa):", {k: int((inc_['n_lda_tokens'] >= k).sum()) for k in (2, 3, 4, 5, 6)})

rt = pd.DataFrame({"semua": df.groupby("source_month").size(),
                   "ikut_BERTopic": df[df["in_topic_corpus"]].groupby("source_month").size(),
                   "ikut_LDA": df[df["in_lda_corpus"]].groupby("source_month").size()}).fillna(0).astype(int)
rt["BERTopic_%"] = (100 * rt["ikut_BERTopic"] / rt["semua"]).round(1)
rt["LDA_%"] = (100 * rt["ikut_LDA"] / rt["semua"]).round(1)
rt["balasan_di_data_%"] = (100 * df[df["in_topic_corpus"]].groupby("source_month")["is_reply"].apply(lambda s: s.eq("True").mean())).round(1)
print("\nRetensi per bulan (pembagi proporsi topik di RQ3):"); display(rt)
for col in ("BERTopic_%", "LDA_%"):
    gap = rt[col].max() - rt[col].min()
    print(f"{col}: selisih antarbulan {gap:.1f} poin" + ("> 10 poin: lihat solusi di dokumentasi RQ3" if gap > 10 else "  (aman)"))

Stemming 19,978 kata unik ...
Frasa terbentuk: 28 → ['baca_buku', 'bentuk_karakter', 'diri_sendiri', 'gen_alpha', 'ikut_campur', 'ikut_kelas', 'kasih_sayang', 'luar_biasa', 'mama_papa', 'masa_depan', 'masa_lalu', 'media_sosial', 'mental_health', 'minta_maaf', 'nikita_willy', 'orang_tua', 'pola_pikir', 'rumah_tangga', 'salah_satu', 'sejak_dini']

Distribusi jumlah token LDA per post (data):


,count,mean,std,min,25%,50%,75%,max
n_lda_tokens,7360.0,14.2,7.8,2.0,7.0,14.0,21.0,47.0


Dengan MIN_LDA_TOKENS = 3: 7,302 post ikut LDA (58 dikeluarkan). Naikkan ambang bila median token rendah.
Ambang lain (post data yang tersisa): {2: 7360, 3: 7302, 4: 6980, 5: 6560, 6: 6123}

Retensi per bulan (pembagi proporsi topik di RQ3):


,semua,ikut_BERTopic,ikut_LDA,BERTopic_%,LDA_%,balasan_di_data_%
source_month,,,,,,
2025-10,678,616,609,90.9,89.8,41.1
2025-11,652,589,583,90.3,89.4,40.2
2025-12,660,583,575,88.3,87.1,39.8
2026-01,681,613,605,90.0,88.8,40.3
2026-02,684,624,624,91.2,91.2,45.2
2026-03,691,632,627,91.5,90.7,44.8
2026-04,689,643,642,93.3,93.2,47.0
2026-05,652,587,582,90.0,89.3,40.5
2026-06,655,605,601,92.4,91.8,45.6


BERTopic_%: selisih antarbulan 5.0 poin  (aman)
LDA_%: selisih antarbulan 6.1 poin  (aman)


## 4. Cek mutu dan cakupan slang
Kata yang belum dikenal ditulis ke `oov_candidates.csv`. 

In [5]:
inc = df[df["in_topic_corpus"]]
print("Contoh (asli → text_embed → text_tokens):")
for _, r in inc.sample(min(8, len(inc)), random_state=SEED).iterrows():
    print(f"\n  asli  : {r['text'][:200]}\n  embed : {r['text_embed'][:200]}\n  tokens: {r['text_tokens'][:200]}")

print("\nKata terpopuler di data (sebelum stemming):")
top_raw = Counter(w for ts in raw_tokens[df["in_topic_corpus"]] for w in ts).most_common(40)
display(pd.DataFrame(top_raw, columns=["kata", "jumlah"]).T)

print("Kata terpopuler di text_tokens:")
top_lda = Counter(w for ts in tokens[df["in_topic_corpus"]] for w in ts).most_common(30)
display(pd.DataFrame(top_lda, columns=["kata", "jumlah"]).T)

print("Distribusi panjang semua post vs data:")
display(pd.DataFrame({"semua": df["n_tokens"].describe(), "data": inc["n_tokens"].describe()}).round(1).T)

# cakupan kosakata
def is_known(w):
    return (w in LEMMAS) or (w in STOP) or (w in KNOWN_EXTRA) or (w in STEM_FIX) or ("-" in w and all(p in LEMMAS or stem_word(p) in LEMMAS for p in w.split("-"))) \
           or (stem_word(w) in LEMMAS)
all_tok = [w for t in inc["text_embed"] for w in t.split() if len(w) > 1]
type_counts = Counter(all_tok)
known_types = {w for w in type_counts if is_known(w)}
tok_cov = sum(type_counts[w] for w in known_types) / sum(type_counts.values())
print(f"\nCakupan kosakata: {tok_cov:.1%} dari kata (token) dan {len(known_types) / len(type_counts):.1%} dari jenis kata (tipe) dikenali.")

# cakupan per bulan
known_cache = {w: (w in known_types) for w in type_counts}
rows = []
for m, g in inc.groupby("source_month"):
    toks = [w for t in g["text_embed"] for w in t.split() if len(w) > 1]
    rows.append({"source_month": m, "n_token": len(toks),
                 "cakupan_%": round(100 * float(np.mean([known_cache[w] for w in toks])), 1) if toks else np.nan})
cov_month = pd.DataFrame(rows).set_index("source_month")
print("Cakupan kosakata per bulan:"); display(cov_month.T)
cov_gap = cov_month["cakupan_%"].max() - cov_month["cakupan_%"].min()
if cov_gap > 3:
    print(f"Cakupan antarbulan berselisih {cov_gap:.1f} poin. Bulan terendah mungkin punya slang baru: "
          "lihat oov_candidates.csv dan perluas slang_extra.csv.")

oov = Counter({w: c for w, c in type_counts.items() if w not in known_types})
ctx = {}
for t in inc["text_embed"]:
    for w in set(t.split()):
        if w in oov and w not in ctx:
            ctx[w] = t[:90]
cand_rows = [(w, c, ctx.get(w, ""), "") for w, c in oov.most_common(300) if c >= 2]
oov_df = pd.DataFrame(cand_rows, columns=["slang", "jumlah", "contoh", "baku"])
oov_df.to_csv(OUT_DIR / "oov_candidates.csv", index=False, encoding="utf-8-sig")
print(f"Kata belum dikenal (muncul ≥ 2×): {len(oov_df)} jenis → oov_candidates.csv. 30 teratas:")
display(oov_df.head(30)[["slang", "jumlah", "contoh"]])

Contoh (asli → text_embed → text_tokens):

  asli  : •[#.tag : Few tweet AU ; Fake Chat ; Jawa, jowo dan jiwo (jiwo sopo?) ; Slice of Life ; Harsh words (misuh²) ; Slengean ; parenting VOC ; horror *soon, jomok romantic *soon ; Ryulnyul/Rulyul *soon ; s
  embed : tag few tweet au fake chat jawa jowo dan jiwo jiwo sopo slice of life harsh words misuh slengean parenting voc horror soon jomok romantic soon ryulnyul rulyul soon sembarang
  tokens: tag tweet au fake chat jawa jowo jiwo jiwo sopo slice life harsh words misuh slengean voc horror soon jomok romantic soon ryulnyul rulyul soon sembarang

  asli  : habis join kelas parenting krn ngegantiin mama😧
  embed : habis join kelas parenting karena menggantikan mama
  tokens: habis join kelas ganti mama

  asli  : penasaran bgt dulu pola asuh ortunya dia tuh gimana ya sampe gabisa mikir oh ini gw lgi pidato formal minimal jaga omongan lu tolol https://t.co/D0l1CteHRK
  embed : penasaran sangat dulu pola asuh orang tuanya dia tuh bagaimana 

,0,1,2,3,4,5,6,7,8,9,...,30,31,32,33,34,35,36,37,38,39
kata,anak,orang,tua,gentle,voc,belajar,ibu,anaknya,salah,dulu,...,baca,terlalu,ni,tetap,bagus,hari,diri,mama,zaman,ternyata
jumlah,3822,2551,1382,1045,722,627,612,596,535,471,...,237,224,224,218,212,211,208,207,205,204


Kata terpopuler di text_tokens:


,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
kata,anak,orang_tua,gentle,ajar,orang,ibu,voc,benar,salah,baik,...,sendiri,suka,ilmu,bapak,beri,pikir,jg,hidup,rumah,laku
jumlah,4643,1781,1045,969,911,805,722,684,610,564,...,359,336,321,310,305,303,296,287,280,275


Distribusi panjang semua post vs data:


,count,mean,std,min,25%,50%,75%,max
semua,8072.0,24.0,14.3,1.0,11.0,22.0,37.0,61.0
data,7360.0,25.9,13.5,5.0,13.0,25.0,38.0,61.0



Cakupan kosakata: 86.0% dari kata (token) dan 48.8% dari jenis kata (tipe) dikenali.
Cakupan kosakata per bulan:


source_month,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
n_token,16416.0,14761.0,14125.0,16360.0,15977.0,17030.0,16458.0,14996.0,16253.0,15819.0,16102.0,15121.0
cakupan_%,85.6,84.9,85.2,85.9,86.1,86.5,86.0,86.3,86.5,86.6,86.3,86.0


Kata belum dikenal (muncul ≥ 2×): 300 jenis → oov_candidates.csv. 30 teratas:


,slang,jumlah,contoh
0,gentle,1045,kalau kayak begini ngasitaunya harus pakai gentle parenting atau voc parenting serius bert
1,voc,722,kalau kayak begini ngasitaunya harus pakai gentle parenting atau voc parenting serius bert
2,jg,296,pentingnya kita memilih bibit bebet bobot yang baik jangan terlalu terburu untuk memutuska
3,ni,224,anak suka main handphone apa kata cuba bagi aktiviti fun bermanfaat worksheet comel ni ban
4,tu,192,selama jokes hidup di rumah tu enak daripada ngekos rantau tapi bayarnya pakai mental maka
5,ku,166,semua akan pret pada waktunya pokoknya mah apa itu gentle parenting akan ku tangani nana d
6,co-parenting,136,taur mereka aslinya sangat lovey dovey bahagia sejahtera gembira kok mereka co-parenting b
7,parents,123,dari kamar hanya bisa kepake kamar is another level of hoardering setelah saya turun tanga
8,strict,120,jujur saya pintar menyembunyikan emosiku sendiri dikarenakan di mata ibuku saya harus teta
9,soft,95,iya ini ril parenting soft spoken umi abi saya anak mereka


## 5. Simpan hasil
- `topic_preprocess_output.csv`: semua baris `relevant.csv` + kolom baru 
- `dup_of`: untuk baris duplikat, `id` post wakilnya; kosong untuk baris lain. Dipakai di notebook model topik agar duplikat mewarisi topik wakilnya
- `preprocess_report.json`: untuk lihat keterbatasan data
- `slang_dict.csv`, `stopwords.txt`, `stopwords_noquery.txt`: kamus yang dipakai
- `oov_candidates.csv`: kata yang belum dikenal; bahan memperluas kamus

In [6]:
NEW_COLS = ["text_embed", "text_tokens", "text_tokens_noquery", "n_tokens", "n_content", "n_lda_tokens",
            "n_lda_tokens_noquery", "in_topic_corpus", "in_lda_corpus", "exclude_reason", "dup_of"]
out = df[[c for c in df.columns if c not in NEW_COLS] + NEW_COLS].copy()
out.to_csv(OUT_DIR / "topic_preprocess_output.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(sorted(SLANG.items()), columns=["slang", "baku"]).to_csv(OUT_DIR / "slang_dict.csv", index=False, encoding="utf-8-sig")
(OUT_DIR / "stopwords.txt").write_text("\n".join(sorted(STOP)), encoding="utf-8")
(OUT_DIR / "stopwords_noquery.txt").write_text("\n".join(sorted(STOP | QUERY_TERMS)), encoding="utf-8")

inc = df[df["in_topic_corpus"]]
report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "input_rows": len(df),
    "settings": {"min_tokens": MIN_TOKENS, "min_content": MIN_CONTENT, "min_lda_tokens": MIN_LDA_TOKENS,
                 "use_stemming": USE_STEMMING, "use_bigrams": USE_BIGRAMS, "bigram_min_count": BIGRAM_MIN_COUNT,
                 "bigram_threshold": BIGRAM_THRESHOLD, "use_lexicon": USE_LEXICON, "use_affix_rules": USE_AFFIX_RULES,
                 "domain_stopwords": sorted(DOMAIN_WORDS) + ["pola asuh"], "query_terms": sorted(QUERY_TERMS), "seed": SEED},
    "slang": {"curated": n_curated, "slang_extra_csv": n_extra, "public_lexicon_entries": len(LEX),
              "token_coverage": round(float(tok_cov), 4), "type_coverage": round(len(known_types) / len(type_counts), 4),
              "oov_types_ge2": int(len(oov_df)),
              "token_coverage_by_month": {k: float(v) for k, v in cov_month["cakupan_%"].items()}},
    "retention_by_month": rt.to_dict(orient="index"),
    "corpus": {"in_topic_corpus": int(len(inc)), "in_lda_corpus": int(df["in_lda_corpus"].sum()),
               "excluded_by_reason": {k: int(v) for k, v in df.loc[~df["in_topic_corpus"], "exclude_reason"].value_counts().items()},
               "in_corpus_by_month": {k: int(v) for k, v in inc["source_month"].value_counts().sort_index().items()},
               "in_corpus_by_group": {k: int(v) for k, v in inc["group"].value_counts().items()},
               "median_tokens": float(inc["n_tokens"].median()),
               "median_lda_tokens": float(inc["n_lda_tokens"].median()),
               "bigrams_found": bigrams_found[:100]},
    "notes": [
        "Teks asli (text) tidak diubah; semua hasil ada di kolom baru.",
        "Tidak ada baris dibuang: post yang tidak cocok untuk topic modeling ditandai in_topic_corpus = False.",
        "BERTopic memakai text_embed (stopword tidak dibuang, tanpa stemming); LDA memakai text_tokens.",
        "Stopword domain (parenting, pola asuh) hanya dibuang di cabang LDA; kata khas gaya dipertahankan.",
        "Istilah pencarian (gentle, voc, ...) bisa mendominasi topik karena kuota sampel; tersedia versi tanpa istilah itu.",
        "Slang dinormalisasi dengan kamus sendiri + leksikon publik + aturan imbuhan; cakupan dilaporkan di bagian slang.",
        "Ambang kosakata (min_df/max_df) ditentukan di notebook model topik.",
    ],
}
with (OUT_DIR / "preprocess_report.json").open("w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

assert len(out) == len(df) and out["id"].is_unique, "Jumlah baris/ID tidak konsisten"
assert (inc["text_embed"].str.len() > 0).all(), "Ada post di data dengan text_embed kosong"
assert (inc["n_tokens"] >= MIN_TOKENS).all() and (inc["n_content"] >= MIN_CONTENT).all(), "Ambang data tidak terpenuhi"
assert not inc["text_embed"].duplicated().any(), "Masih ada duplikat di data"
assert df.loc[df["dup_of"] != "", "dup_of"].isin(set(inc["id"])).all(), "dup_of menunjuk ke post yang tidak ada di data"
assert (df["dup_of"] != "").sum() == (df["exclude_reason"] == "dup_after_clean").sum(), "dup_of tidak cocok dengan exclude_reason"
assert (df.loc[df["in_lda_corpus"], "n_lda_tokens"] >= MIN_LDA_TOKENS).all() and not (df["in_lda_corpus"] & ~df["in_topic_corpus"]).any()
print(f"topic_preprocess_output.csv : {len(out):,} baris")
print(f"duplikat terhubung ke wakilnya lewat dup_of: {(df['dup_of'] != '').sum():,} baris")
print("preprocess_report.json, slang_dict.csv, stopwords.txt, stopwords_noquery.txt, oov_candidates.csv")
print("Semua pemeriksaan integritas lolos")

topic_preprocess_output.csv : 8,072 baris
duplikat terhubung ke wakilnya lewat dup_of: 1 baris
preprocess_report.json, slang_dict.csv, stopwords.txt, stopwords_noquery.txt, oov_candidates.csv
Semua pemeriksaan integritas lolos
